# Lab 3: The geometry of two views and 3D reconstruction

In this lab session, we are going to compute the 3D position of a pair of cameras and a set of keypoints.  For this, we will first compute correspondences between the two images.  Then, we will robustly compute the Fundamental matrix that encodes the geometry of the two views. From the Fundamental matrix and the camera calibration matrix -- that we learnt to estimate in the third lab session -- we will get the Essential matrix, which encodes the relative motion between the two cameras.  We will then compute the motion between the cameras and, finally, triangulate the matched keypoints to obtain their 3D position.

The goals of this lab assignment are the following:

- How to estimate the fundamental matrix that relates two images, corresponding to two different views of the same scene, given a set of correspondences between them. In particular, we will use the 8-point algorithm.

- How to compute the relative pose of two calibrated cameras

- How to triangulate point matches to reconstruct their 3D position.


### Deliverables:
1. **Completed Jupyter Notebook (GitHub Classroom)**:
Complete the provided starter code by filling in the missing implementation details. You must submit a fully-executed Jupyter Notebook (.ipynb) where all cells have been run and the outputs (images, logs, or results) are clearly visible.

2. **Completed Questionnaire (Gradescope)**:
Complete the accompanying questionnaire as you work through the code. It is recommended to fill this out during execution to ensure your answers align with the observed results in your notebook.


## **1. Estimation of the fundamental matrix**

### **1.1 The 8-point algorithm**

The first task is to create the function that estimates the fundamental matrix given a set of point correspondences between a pair of images. We provide the incomplete function `fundamental_matrix` that computes,
with the normalised 8-point algorithm, an estimation of the Fundamental matrix
from a set of tentative point correspondences.  


In [ ]:
import logging
import math
import random
import sys

import cv2
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import Markdown
from matplotlib import pyplot as plt
from lab_utils import plot_camera

from tqdm.notebook import tqdm


In [ ]:
def Normalization(image_points):
    # TODO: Compute the normalization matrix T and the noramlized_image_points

    return T, normalized_image_points


def eight_point_algorithm(x_i_array, xp_i_array):
    """
    Estimate the fundamental matrix using the 8-point algorithm as in seminar 3.
    Hint: Remember that it can be computed using 8 or more correspondences.
    """
    # TODO: Normalize points in both images

    # TODO: compute the fundamental matrix F
    F = ...


    return F

### Test your algorithm!
The code below contains a toy example where we know the ground truth correspondences and ground-truth fundamental matrix. Use the code to test that the completed function is working properly.

In [ ]:
def test_eight_point_algorithm(number_of_correspondences=8):
    # Two camera matrices for testing purposes
    # Camera 1 in the origin
    P1 = np.zeros((3, 4))
    P1[:,:3] = np.identity(3)

    # Camera 2 is a rotation of 15 degrees around the y-axis and a translation of [0.3, 0.1, 0.2]
    theta = np.radians(15)
    R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])
    t = np.array([0.3, 0.1, 0.2])
    P2 = np.column_stack((R, t))

    # Generate random points in front of the camera
    N = number_of_correspondences
    X = np.vstack([
        np.random.uniform(0, 1, (1, number_of_correspondences)),
        np.random.uniform(0, 0.5, (1, number_of_correspondences)),
        np.random.uniform(0, 3, (1, number_of_correspondences)),
        np.ones((1, number_of_correspondences))
    ]) # point in the 3D space in homogeneous coordinates

    # Project to obtain the points in homogenious coordinates in the projective space of dimension 2
    x1_test_h = P1 @ X
    x2_test_h = P2 @ X

    # Estimate fundamental matrix
    F_es = eight_point_algorithm(x1_test_h, x2_test_h)

    # Ground truth fundamental matrix
    t_x = np.array([[0, -t[2], t[1]], 
                    [t[2], 0, -t[0]], 
                    [-t[1], t[0], 0]])
    F_gt = t_x @ R

    # Normalize for comparison
    F_gt /= np.linalg.norm(F_gt)
    F_es /= np.linalg.norm(F_es)
    
    # Check for sign flip
    if np.sign(F_gt[0,0]) != np.sign(F_es[0,0]):
        F_es *= -1

    # Evaluation: these two matrices should be very similar
    dif_norm = np.linalg.norm(F_gt - F_es)

    # Print results
    logging.info(
        ("\n" * 2).join([f"\n\nGround truth F: {F_gt}", f"Estimated F: {F_es}", f"Norm of the difference: {dif_norm}"])
    )

    if dif_norm < 1e-5:
        display(
            Markdown(
                f'<span style="color: #00ff00">The function `fundamental_matrix()` with `{number_of_correspondences}` correspondences seems correct!<br>'
                f"Norm of the difference between estimated and ground truth matrix is {dif_norm:.3g}; i.e. almost zero.</span>"
            )
        )
    else:
        display(
            Markdown(
                f'<span style="color: #ff0000">The function `fundamental_matrix()` with `{number_of_correspondences}` correspondences might be wrong!<br>'
                f"Norm of the difference between estimated and ground truth matrix is {dif_norm:.3g}; i.e. different from zero.</span>"
            )
        )

test_eight_point_algorithm(number_of_correspondences=8)
test_eight_point_algorithm(number_of_correspondences=20)

### **1.2 Robust estimation of the fundamental matrix**

The next step is to robustly compute the Fundamental matrix from the point correspondences. For that we will use the function `ransac_fundamental_matrix` that you have to complete.

The next code computes the image correspondences.

In [ ]:
# Read images
img1_bgr = cv2.imread("Data/manikin1.jpg", cv2.IMREAD_COLOR)
img2_bgr = cv2.imread("Data/manikin2.jpg", cv2.IMREAD_COLOR)
img1_rgb = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15, 8))
plt.suptitle("Images of the same object from different viewpoints")
for i, image in enumerate([img1_rgb, img2_rgb], start=1):
    plt.subplot(1, 2, i)
    plt.title(f"Image {i}")
    plt.imshow(image)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
%%time
# Read images
img1_gray = cv2.imread("Data/manikin1.jpg", cv2.IMREAD_GRAYSCALE)
img2_gray = cv2.imread("Data/manikin2.jpg", cv2.IMREAD_GRAYSCALE)

# Initiate SIFT detector
sift = cv2.SIFT_create(3000)

# find the keypoints and descriptors
kp1, des1 = sift.detectAndCompute(img1_gray, None)
kp2, des2 = sift.detectAndCompute(img2_gray, None)

# Keypoint matching
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches = bf.match(des1, des2)

# Filter matches based on distance
match_distance_threshold = 500
matches = [m for m in matches if m.distance < match_distance_threshold]

# Show matches
img_12 = cv2.drawMatches(
    img1_rgb, kp1, img2_rgb, kp2, matches, None, 12, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(18.5, 10.5))
plt.title("SIFT correspondences between images")
plt.imshow(img_12)
plt.axis("off")
plt.show()


**Q4.** Complete the function `ransac_fundamental_matrix` below.

In [ ]:
def compute_inliers(F, x1, x2, th):
    """
    Computes inliers for a Fundamental Matrix using Sampson Distance.
    
    Args:
        F: Fundamental Matrix (3x3)
        x1: Homogeneous coordinates in image 1 (3xN)
        x2: Homogeneous coordinates in image 2 (3xN)
        th: Error threshold (float)
        
    Returns:
        inliers_indices: 1D array of indices where the error < th
    """
    # TODO: Compute the Sampson distance
    numerator = ... # shape (N,)
    denominator = ... # shape (N,)
    sampson_distance = ... # shape (N,)

    # TODO: 5. Find and return indices where distance < th

    return inliers_indices


def ransac_eight_point_algorithm(x1, x2, th, min_iterations):
    """
    RANSAC algorithm to estimate the fundamental matrix using the 8-point algorithm.
    """
    Ncoords, Npts = x1.shape

    p = 0.99
    # TODO: Select the number of correspondences to sample
    s = ...

    it = 0
    N = min_iterations
    best_inliers = np.empty(1)
    best_N = 1e16

    # Initialize tqdm with dynamic total
    pbar = tqdm(total=N, desc="RANSAC iterations", unit="iter")
    
    while it < min_iterations or it < best_N:

        # TODO: Randomly select s correspondences to compute F and its inliers
        inliers = ...


        # update estimate of iterations (the number of trials) to ensure we pick, with probability p,
        # an initial data set with no outliers
        num_inliers = inliers.shape[0]
        w = num_inliers / Npts
        pOutlier = 1 - w**s
        # avoid pOutlier to be 0 or 1 as it causes a log(0) or division by 0
        pOutlier = max(pOutlier, sys.float_info.epsilon)
        pOutlier = min(pOutlier, 1 - sys.float_info.epsilon)
        estimated_N = math.log(1 - p) / math.log(pOutlier)

        prev_best_N = best_N
        best_N = min(best_N, estimated_N)  # Keep the best (lowest) N

        # TODO: Update the best inliers

        if len(inliers) == len(best_inliers):
            print(f"it:{it} - best inliers: {len(best_inliers)} - w: {w:.2f} - N: {best_N:.5g} - prev_best_N: {prev_best_N:.5g}")

        if int(N) > pbar.total:
            pbar.total = int(N)
            pbar.refresh()

        pbar.update(1)
        it += 1

    pbar.close()

    print(f"Number of iterations: {it}, best inliers: {len(best_inliers)} - N: {best_N:.5g}")

    # Recompute F from all the inliers
    F = eight_point_algorithm(x1[:, best_inliers], x2[:, best_inliers])
    inliers_recomputed = compute_inliers(F, x1, x2, th)
    print(f"Recomputed inliers: {len(inliers_recomputed)}")

    return F, best_inliers

In [ ]:
%%time
# Robust estimation of the fundamental matrix
x1 = []
x2 = []
for m in matches:
    x1.append([kp1[m.queryIdx].pt[0], kp1[m.queryIdx].pt[1], 1])
    x2.append([kp2[m.trainIdx].pt[0], kp2[m.trainIdx].pt[1], 1])

x1 = np.asarray(x1).T # shape (3, N)
x2 = np.asarray(x2).T # shape (3, N)

F, indices_inlier_matches = ransac_eight_point_algorithm(x1, x2, th=2, min_iterations=2000)
inlier_matches = [matches[i] for i in indices_inlier_matches]

# From now on, we will work with the inlier matches only.
x1 = x1[:, indices_inlier_matches]
x2 = x2[:, indices_inlier_matches]

In [ ]:
img_12 = cv2.drawMatches(
    img1_rgb, kp1, img2_rgb, kp2, inlier_matches, None, 12, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)
plt.figure(figsize=(18.5, 10.5))
plt.title("Inliers of SIFT correspondences based on the estimated fundamental matrix (F)")
plt.imshow(img_12)
plt.axis("off")
plt.show()

### **1.3 Epipolar lines**

We will now visualize the epipolar lines associated to some points.

**Q5.** Choose 3 inlier points in the first image and compute their corresponding epipolar lines (in homogeneous coordinates) in the second image. Do the same but now choosing points in the second image and showing their epipolar lines in the first image. Provide the commands you used for that. The commands for showing the lines on top of the image are already provided.

In [ ]:
# TODO: compute the epipolar lines
l2 = ... # epipolar lines in image 2
l1 = ... # epipolar lines in image 1

# choose three random (inlier) correspondences
num_inliers = x1.shape[1]
random.seed(42)
chosen_indices = random.sample(range(1, num_inliers), k=3)

# Function required to display lines
def get_line_points(l, I):
    x = np.arange(0, np.array(I).shape[1])
    y = (-l[0] * x - l[2]) / l[1]
    return x, y


# Display image with epipolar lines and points
plt.figure(figsize=(18.5, 10.5))
points_color = "black"
for num_plot, (image, epipolar_lines, points) in enumerate(
    zip([img1_rgb, img2_rgb], [l1, l2], [x1, x2]), start=1
):
    plt.subplot(1, 2, num_plot)
    plt.title(f"Image {num_plot} with points and computed epipolar lines")
    for i, index in enumerate(chosen_indices, start=1):
        plt.plot(*get_line_points(epipolar_lines[:, index], image), label=f"$l{chr(39) * (num_plot == 2)}_{i}$")

    for i, (x, y) in enumerate(zip(points[0, chosen_indices], points[1, chosen_indices]), start=1):
        plt.scatter(x=x, y=y, c=points_color, s=30, zorder=10)
        plt.text(
            x,
            y,
            f"$x{chr(39) * (num_plot == 2)}_{i}$",
            fontsize=20,
            ha="right",
            va="bottom",
            color=points_color,
            weight="bold",
        )
    plt.imshow(image)
    plt.legend()
plt.show()

**Q6.** Epipolar lines and epipoles.
1. Find the intersection point where the epipolar lines cross. 
2. Compute the epipoles from the Fundamental matrix.
3. Compare them

Then:

4. Compute the epipolar lines as the line that passes through the matching point and the epipole.
5. Compare them to the previous epipolar lines (the ones obtained by applying the fundamental matrix to the matched point).

## 2. Structure computation

### **2.1 Triangulation**

Before recovering the camera motion, we will first need to implement a function to triangulate matches.  This is a function that takes as input a match and the projection matrices of two cameras, and computes the 3D position of a match.

We want then to find the 3D point $\vec{X}$ such that its projection onto the images are $\vec{x} = P \vec{X}$ and $\vec{x'} = P' \vec{X}$ (being $P$ and $P'$ the projection matrices of each camera).


**Q7.** Derive the expression of matrix $A$ in the system of equations in algebraic form, $A \mathbf{X} = \mathbf{0}$, that we need to solve in order to estimate the homogeneous coordinates of the 3D point $\mathbf{X}$.

*Answer here. You can write it on a paper and show a picture here, or write it in LaTex.*


**Q8.** Complete the code of the `triangulate`function below.

In [ ]:
def triangulate(x1, x2, P1, P2, imsize=img1_rgb.shape):
    """
    Triangulate points from two images.
    """
    # Adapting the input to the expected format
    if x1.ndim == 1:
        x1 = np.array([x1]).T
        x2 = np.array([x2]).T

    # Check the dimensions of the input matrices
    assert x1.shape[0] == 3 and x2.shape[0] == 3 and x1.shape == x2.shape, "x1 and x2 must be 3xN matrices"
    assert P1.shape == (3, 4) and P2.shape == (3, 4), "P1 and P2 must be 3x4 matrices"

    # Number of points to triangulate and image dimensions
    n_points = x1.shape[1]
    num_pixels_x = imsize[0]
    num_pixels_y = imsize[1]

    # Normalize the points by dividing by their third coordinate
    x1 = x1 / x1[2, :]
    x2 = x2 / x2[2, :]

    # Normalize image coordinates from pixel space to approximately [-1, 1]
    # to improve numerical conditioning in triangulation.
    H = [[2 / num_pixels_x, 0, -1], [0, 2 / num_pixels_y, -1], [0, 0, 1]]

    # Apply the same normalization to points and camera matrices
    # so the projection geometry remains consistent.
    x1_norm = H @ x1
    x2_norm = H @ x2
    P1_norm = H @ P1
    P2_norm = H @ P2

    # TODO: compute the triangulated points
    X = ...


    return X # X is a 4xN matrix where each column is a 3D point in homogeneous coordinates

You may use the following test code with a toy example to validate that the `triangulate`function is working properly.
In summary, this code simulates the process of 3D point projection, triangulation, and evaluation of the reprojection error in a scenario with two cameras and randomly generated 3D points.

In [ ]:
def test_triangulate_function(number_of_correspondences):
    # Two camera matrices for testing purposes
    # Camera 1 in the origin
    P1 = np.zeros((3, 4))
    P1[:,:3] = np.identity(3)

    # Camera 2 is a rotation of 15 degrees around the y-axis and a translation of [0.3, 0.1, 0.2]
    theta = np.radians(15)
    R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])
    t = np.array([0.3, 0.1, 0.2])
    P2 = np.column_stack((R, t))

    # Generate random points in front of the camera
    X = np.vstack([
        np.random.uniform(0, 1, (1, number_of_correspondences)),
        np.random.uniform(0, 0.5, (1, number_of_correspondences)),
        np.random.uniform(0, 3, (1, number_of_correspondences)),
        np.ones((1, number_of_correspondences))
    ]) # point in the 3D space in homogeneous coordinates

    # Project to obtain the points in homogenious coordinates in the projective space of dimension 2
    x1_test_h = P1 @ X
    x2_test_h = P2 @ X

    # Estimate the 3D points (you need to create this function)
    # The function triangulate() is called to estimate the 3D coordinates of the points using the corresponding 2D points and camera matrices
    X_trian = triangulate(x1_test_h, x2_test_h, P1, P2, ((2, 2)))

    # Evaluation: compute the reprojection error
    # The synthetic 3D points (X_eucl) and the triangulated 3D points (x_cart) are converted from homogeneous coordinates to Cartesian coordinates.
    X_trian_cart = X_trian / X_trian[3, :]
    X_gt_cart = X / X[3, :]
    # The difference between these two sets of points is computed and printed as the reprojection error.
    reproj_err = np.linalg.norm(X_gt_cart - X_trian_cart, axis=0)
    print(f"Reprojection error for n={number_of_correspondences} points: mean={reproj_err.mean():.3g}, std={reproj_err.std():.3g}", sep="\n", end="\n" * 2)
    if np.allclose(reproj_err, 0, atol=1e-5):
        display(Markdown('<span style="color: #00ff00">The function `triangulate()` seems correct!</span>'))
    else:
        display(Markdown('<span style="color: #ff0000">The function `triangulate()` might be wrong!</span>'))

In [ ]:
test_triangulate_function(number_of_correspondences=10)
test_triangulate_function(number_of_correspondences=100)

### 2.2 Relative motion between the two cameras
As seen in the theory class, the relative position between the cameras can be computed from the Essential matrix, $E$. In order to estimate $E$, we need the Fundamental matrix, $F$, that we have just computed and the camera calibration matrix, $K$, that we learnt how to estimate during the second lab.

**Q9.** Compute the Essential matrix from the Fundamental matrix and the camera calibration matrix in the code below.

In [ ]:
# K matrix estimated as in lab 2
K = np.array([[3531.97, 9.59, 2304.33], [0, 3537.34, 1751.75], [0, 0, 1]])

# TODO: Complete the essential matrix
E = ...
print(E)

We will assume that the first camera is at the origin, with no rotation, and that the second camera has a rotation and translation with respect to the first one.

**Q10.** Write the camera projection matrix $P$ for the first camera.

In [ ]:
# TODO: Compute the projection matrix of the first camera
P1 = ...

print(P1)

The rotation and translation of the second camera can be computed from the SVD decomposition of $E$. There are four possible solutions.

**Q11.** Complete the code below to compute the four candidate solutions for the second camera projection matrix.

In [ ]:
def find_potential_camera_projections(E, K, P1):
    U, D, Vt = np.linalg.svd(E)

    # The SVD of E has several ambiguities. In particular, U and V may be
    # improper rotations in which case we need to change their sign.
    if np.linalg.det(U) < 0:
        U = -U

    if np.linalg.det(Vt) < 0:
        Vt = -Vt


    # TODO: Find the four potential camera projection matrices for the second camera
    P2_potential = np.empty(shape=(4, 3, 4))

    ny, nx, _ = img1_rgb.shape

    fig = go.Figure()
    fig.update_layout(title="Potential camera projection matrices for the second camera (P2)")
    plot_camera(P1, nx, ny, fig, "Reference camera (P1)")
    for i in range(4):
        plot_camera(P2_potential[i], nx, ny, fig, f"Camera_2-Option_{i+1}")
    fig.show()

    return P2_potential

P2_potential = find_potential_camera_projections(E, K, P1)
for i in range(4):
    print(f"P2_potential[{i}]:", P2_potential[i], sep="\n", end="\n" * 2)

### Selecting the proper camera projection matrix

In [ ]:
def find_the_right_camera_projection(P2_potential, x1, x2, P1, indices_inlier_matches):
    ny, nx, _ = img1_rgb.shape
    for P2_i in P2_potential:

        Xi = triangulate(x1[:, 0], x2[:, 0], P1, P2_i, [nx, ny])
        Xi = Xi / Xi[3, :]

        x1est = P1 @ Xi
        x2est = P2_i @ Xi

        if (x1est[2] > 0) and (x2est[2] > 0):
            P2 = P2_i
            break

    fig = go.Figure()
    fig.update_layout(title="Camera 1 and Selected Camera 2")
    plot_camera(P1, nx, ny, fig, "Camera 1")
    plot_camera(P2, nx, ny, fig, "Camera 2")
    fig.show()

    return P2

P2 = find_the_right_camera_projection(P2_potential, x1, x2, P1, indices_inlier_matches)
print(f"P2: {P2}")

### **2.3 Reconstruction from two views**

Once the proper solution for the relative motion between cameras is chosen, we can triangulate all the matches to get a sparse point cloud. 

**Q13.** Complete the code to triangulate all matches.

Use the provided code to plot the reconstructed points. If everything went fine, you should recognize the sparse points corresponding to the different parts of the manikin body at their corresponding 3D positions.

In [ ]:
def reconstruct_points(x1, x2, P1, P2):
    # TODO: Triangulate all (inlier) matches
    X = ...

    # Render the 3D point cloud
    fig = go.Figure()
    fig.update_layout(title="3D Reconstruction")
    # Plot the cameras
    plot_camera(P1, nx, ny, fig, "Camera 1")
    plot_camera(P2, nx, ny, fig, "Camera 2")
    # Plot the points with the correct color from img1_rgb
    rgb_vals = img1_rgb[x1[1].astype(int), x1[0].astype(int)]
    rgb_vals = [f"rgb({int(r)},{int(g)},{int(b)})" for r, g, b in rgb_vals]
    fig.add_trace(go.Scatter3d(x=X[0, :], y=X[2, :], z=-X[1, :], mode="markers", marker=dict(size=2, color=rgb_vals)))
    fig.show()

    # Return the 3D points
    return X

X = reconstruct_points(x1, x2, P1, P2)

Finally, we are going to compute the reprojection error of the reconstructed points.  This is the distance between the detected keypoints, `x1` and `x2`, and the projection of the reconstructed points.

**Q14.** Complete the code to compute the reprojection error of each match. Plot the histogram of the errors with the provided code. Explain the result.

In [ ]:
def reprojection_errors(x1, x2, P1, P2, X):

    # ... complete

    return (err1 + err2) / 2


err = reprojection_errors(x1, x2, P1, P2, X)

In [ ]:
def plot_reprojection_errors(err):
    sns.set_theme(style="whitegrid")
    ax = sns.displot(err, kde=True, bins=10)
    mean_err = np.mean(err)
    plt.axvline(x=mean_err, color='r', linestyle='--', label=f'Mean: {mean_err:.2f} pixels')
    plt.title("Histogram and Distribution of Reprojection Errors")
    plt.xlabel("Reprojection Error (pixels)")
    plt.ylabel("Counts / Density")
    plt.legend()
    plt.show()

plot_reprojection_errors(err)

## 3. 3D-Reconstruction of your own pair of images
1. **Capture your data**: Take pictures of a home-made example.
    - Move the camera to ensure the images are taken from different positions.
    - Make sure to have objects in the foreground and others in the background so the reconstructed scene can be interpreted.
    - Although you need only 2 images, take 5 images with different distances between them so you can choose the pair that works best.
    - Avoid auto-focus to maintain the same focal length for all images.
2. **3D-Reconstruction**: Apply the complete pipeline of this lab.
3. **Evaluation**: Evaluate the resulting reconstruction and its reprojection error.

Technical recommendations:
- **For meaningful SIFT descriptors**: Choose non-repetitive patterns and consider resizing the image if it has too much resolution.
- **SIFT keypoints matching**: If the RANSAC 8-point algorithm takes too long to compute, consider  filtering the SIFT matches by reducing the `match_distance_threshold`.
- **For K estimation**: Reuse the same `K` matrix from Lab 2 if you are using the same camera and settings; otherwise, estimate the intrinsic parameters of `K` by applying the same procedure as in Lab 2.

## 4. EXTRAS


Complete any of the extra exercises if interested: :)
1. Compare the reconstruction of two different pairs from their 5-image set:
- Pair A: Two images taken very close together (Short Baseline).
- Pair B: Two images taken further apart (Long Baseline).
- Task: Compare the depth (Z) stability.

2. The Scale Ambiguity Challenge
- Exercise: Measure a real object in the scene (e.g., the height of a bottle is 12 cm). Then, calculate the distance between two points on that object in the 3D reconstruction.
- Task: Find the scaling factor `s` needed to convert your 3D reconstruction to centimeters. Why can't the Essential Matrix determine this scale automatically?

3. Your reconstruction only shows the SIFT keypoints (a Sparse reconstruction). How would you go about filling in the gaps to create a solid surface (a Dense reconstruction)?